In [ ]:
!pip install -q numpy matplotlib

## Scenario

## STAGE 1 -- Attention and Transformer Components from Scratch (55 min)

## STEP 1.1 -- Why Attention? Motivation from Sequence Models (10 min)

## STEP 1.2 -- Scaled Dot-Product Attention (18 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention using NumPy.

    Args:
        Q: Queries, shape (..., seq_len_q, d_k)
        K: Keys, shape (..., seq_len_k, d_k)
        V: Values, shape (..., seq_len_k, d_v)
        mask: Optional boolean mask, shape broadcastable to (..., seq_len_q, seq_len_k).
              Positions with True are masked (set to -inf before softmax).

    Returns:
        output: Weighted values, shape (..., seq_len_q, d_v)
        attention_weights: Softmax weights, shape (..., seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)

    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    attention_weights = softmax(scores, axis=-1)
    output = attention_weights @ V
    return output, attention_weights

# Demo: 4-token sequence, d_k = 8
np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")                    # (4, 8)
print(f"Attention weights shape: {weights.shape}")         # (4, 4)
print(f"Weights sum per query (should be ~1.0): {weights.sum(axis=-1)}")

# Visualize the attention matrix
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights, cmap="Blues")
ax.set_xlabel("Key position")
ax.set_ylabel("Query position")
ax.set_title("Attention Weights")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Why scaling matters
d_k_small, d_k_large = 8, 512
np.random.seed(42)
Q_s = np.random.randn(4, d_k_small)
K_s = np.random.randn(4, d_k_small)
Q_l = np.random.randn(4, d_k_large)
K_l = np.random.randn(4, d_k_large)

scores_small = Q_s @ K_s.T
scores_large = Q_l @ K_l.T

print(f"Unscaled scores (d_k=8), std:   {scores_small.std():.2f}")
print(f"Unscaled scores (d_k=512), std: {scores_large.std():.2f}")
print(f"Scaled scores (d_k=512), std:   {(scores_large / np.sqrt(512)).std():.2f}")

## STEP 1.3 -- Multi-Head Self-Attention (15 min)

## STEP 1.4 -- Positional Encoding (10 min)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_positional_encoding(max_len, d_model):
    """Generate sinusoidal positional encoding matrix.

    Returns:
        pe: shape (max_len, d_model)
    """
    pe = np.zeros((max_len, d_model))
    position = np.arange(0, max_len).reshape(-1, 1)
    div_term = np.exp(
        np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model)
    )
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

encoding = sinusoidal_positional_encoding(max_len=50, d_model=64)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(encoding.T, cmap="RdBu", aspect="auto")
ax.set_xlabel("Position")
ax.set_ylabel("Dimension")
ax.set_title("Sinusoidal Positional Encoding (d_model=64)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## STAGE 2 -- Exploring Pre-trained Models: BERT, GPT, T5 (55 min)

## STEP 2.1 -- Transformer Encoder vs. Decoder Recap (8 min)

## STEP 2.2 -- BERT: Bidirectional Understanding (15 min)

## STEP 2.3 -- GPT: Autoregressive Generation (15 min)

## STEP 2.4 -- T5: Text-to-Text Framework (12 min)

## STAGE 3 -- Transfer Learning and Fine-tuning (55 min)

## STEP 3.1 -- Transfer Learning Pipeline (12 min)

## STEP 3.2 -- Fine-tuning for Sentiment Classification (22 min)

## STEP 3.3 -- (Stretch) Parameter-Efficient Fine-tuning: LoRA (10 min)

In [ ]:
# LoRA savings calculation (can run with plain Python)
d, r = 768, 8
full_params = d * d
lora_params = d * r + r * d
print(f"Full weight matrix: {full_params:,} parameters")
print(f"LoRA (rank {r}):    {lora_params:,} parameters ({lora_params/full_params:.1%} of full)")

## STEP 3.4 -- (Stretch) GPT-Style Text Generation Strategies (10 min)